# Imports

In [ ]:
import sklearn
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [ ]:
plt.rcParams['figure.figsize'] = (4, 3)
pd.options.display.float_format = '{:.3f}'.format

# Load Dataset

Cada linha é uma transação (compra)\
Um mesmo cliente pode ter feito várias transações

In [ ]:
file_path = "data/dataset.csv"

In [ ]:
df = pd.read_csv(file_path)

In [ ]:
df.sample(5)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

# Engenheria de Atributos

## Colunas de Data

In [ ]:
df.columns[df.columns.str.contains('date|time')]

In [ ]:
df['order_purchase_timestamp']

In [ ]:
# Converte de object para datetime
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

In [ ]:
# Cria um nova coluna só com a data (sem a hora,min,seg)
df['order_purchase_date'] = df['order_purchase_timestamp'].dt.date

In [ ]:
df['order_purchase_date']

In [ ]:
# Cria um nova coluna invoice date
df['invoice_date'] = df['order_purchase_date'].apply(lambda x: datetime.strftime(x, '%Y-%m-%d'))
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['invoice_date']

Alguns insights

In [ ]:
print('{} transações não tem customer id'.format(df['customer_unique_id'].isnull().sum()))

In [ ]:
print('Data das Transações ocorreram entre {} e {}'.format(df['order_purchase_date'].min(),
                                                         df['order_purchase_date'].max()))

## Agregação dos Dados

Recência = números de dias desde a última compra de um cliente, em relação a última compra global\
Frequência = número de transações realizadas por um cliente\
Monetário = valor total que o cliente gastou

A recency vai ser em relação a última compra global, não sobre a data atual.\
E a última compra global foi em 03/09/2018.

In [ ]:
# soma 1 dia porque senao ultimacompraglobal - ultimacompraclientex pode ser 0
# e essa venda não seria considerada
data_mais_recente_global = df['invoice_date'].max() + timedelta(days = 1)
data_mais_recente_global

In [ ]:
# Exemplo com o cliente C10952
df[df['customer_unique_id'] == 'C10952']

In [ ]:
# Última compra feita pelo cliente C10952
df[df['customer_unique_id'] == 'C10952']['invoice_date'].max()

In [ ]:
# Recência do cliente C10952
data_mais_recente_global - df[df['customer_unique_id'] == 'C10952']['invoice_date'].max()

## Pandas Group By, Agg

https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html

In [ ]:
df_rfm = df.groupby(df['customer_unique_id']).agg(recency = pd.NamedAgg(column='invoice_date',aggfunc=lambda x: (data_mais_recente_global - x.max()).days),
                                                frequency = pd.NamedAgg(column='order_id', aggfunc='count'),
                                                monetary = pd.NamedAgg(column='payment_value', aggfunc='sum'))

In [ ]:
df_rfm

In [ ]:
df_rfm.isna().sum()

In [ ]:
df_rfm['recency'].describe()

In [ ]:
# Frequencia da frequencia
df_rfm['frequency'].value_counts()

In [ ]:
df_rfm['frequency'].describe()

In [ ]:
df_rfm['monetary'].describe()

# Definição dos Recursos RFM

In [ ]:
df_rfm

## Recency

In [ ]:
# Vamos ignorar os outliers
df_rfm['recency'].plot(kind='box')

Vamos dividir os dados de Recency em 4 grupos atribuindo um label a cada observação.

A função range em Python gera uma sequência de números. O uso de range(4, 0, -1) cria uma sequência que começa em 4, termina antes de 0 e decrementa em 1 a cada passo. Portanto, a sequência gerada será: 4,3,2,1

Note que o número 0 não está incluído na sequência, pois o ponto final especificado em range é exclusivo.

In [ ]:
r_labels = range(4,0,-1)

https://pandas.pydata.org/docs/reference/api/pandas.qcut.html\
Quantile-based discretization function.\
Discretize variable into equal-sized buckets based on rank or based on sample quantiles. 

In [ ]:
r_groups = pd.qcut(df_rfm['recency'], q=4, labels=r_labels)
r_groups

In [ ]:
r_groups.value_counts()

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html\
Assign new columns to a DataFrame

In [ ]:
df_rfm = df_rfm.assign(R = r_groups.values)
df_rfm

## Frequency

In [ ]:
df_rfm['frequency'].describe()

In [ ]:
df_rfm['frequency'].plot(kind='box')

In [ ]:
df_rfm['frequency'].value_counts()

Você tem a maioria esmagadora dos clientes que compraram apenas 1 vez e alguns poucos que compraram mais de uma vez. Então iremos dividir em 2 grupos apenas.

In [ ]:
def frequency_label(x):
    if x == 1:
        return 1
    if x > 1:
        return 2

In [ ]:
df_rfm['F'] = df_rfm['frequency'].apply(frequency_label)

In [ ]:
df_rfm['F'].value_counts()

In [ ]:
df_rfm

### Método Alternativo

Fórmula do Rank
Rank(x) = número de elementos anteriores a x + [(k + 1) / 2]\
k = numero de vezes que x aparece

In [ ]:
# Rank de cada elemento da coluna Frequency
# Valores repetidos ele faz a média
# numero de elementos anteriores = 0 + (k + 1) / 2 , k = numero de vezes que o elemento aparece
df_rfm['frequency'].rank()

Rank(x) em percentil = rank(x) / n

In [ ]:
df_rfm['frequency'].rank(pct = True)

In [ ]:
# Maneira alternativa
# f = lambda x: (edges >= x).values -> retorna [c c c] em que c = [Falso, Verdadeiro]
# f = lambda x: (edges >= x).values.argmax() retorna o argmax que sempre é 1 ou 2.
# porque x nunca é zero

def user_defined_q_cut(series, edges):   
    f = lambda x: (edges >= x).values.argmax()
    
    return series.rank(pct = True).apply(f)

In [ ]:
n = 2
edges = pd.Series(float(i) / n for i in range(n+1))
edges

In [ ]:
# argmax ou argmin retorna o primeiro indice
# [True True True] = [1 1 1]
pd.Series(data=[True,True,True]).argmax()

Como eu quero que seja label 1 ou 2\
ele criou 3 edges e o primeiro (argmax 0) é sempre Falso\
Então sempre vai retornar argmax 1 ou argmax 2.

In [ ]:
user_defined_q_cut(series = df_rfm['frequency'], edges = edges)

## Monetary

In [ ]:
df_rfm['monetary'].describe()

In [ ]:
df_rfm['monetary'].plot(kind='box')

In [ ]:
# Dividir a coluna 'Monetary' em 4 grupos
m_labels = range(1,5)

In [ ]:
m_groups = pd.qcut(df_rfm['monetary'], q=4, labels = m_labels)

In [ ]:
m_groups.value_counts()

In [ ]:
df_rfm = df_rfm.assign(M = m_groups.values)

In [ ]:
df_rfm

# Processamento de Dados

In [ ]:
df_rfm.info()

In [ ]:
df_rfm['R'] = pd.to_numeric(df_rfm['R'])

In [ ]:
df_rfm['F'] = pd.to_numeric(df_rfm['F'])

In [ ]:
df_rfm['M'] = pd.to_numeric(df_rfm['M'])

In [ ]:
df_rfm.info()

In [ ]:
df_rfm

# Criação dos Segmentos RFM

In [ ]:
df_rfm[['R','F','M']].sum(axis=1)

In [ ]:
df_rfm['Score_RFM'] = df_rfm[['R','F','M']].sum(axis=1)

In [ ]:
df_rfm

In [ ]:
def concatenate(x):
    return str(int(x['R'])) + str(int(x['F'])) + str(int(x['M']))

In [ ]:
df_rfm['Segmento_RFM'] = df_rfm.apply(concatenate, axis=1)

In [ ]:
df_rfm

In [ ]:
df_rfm['Segmento_RFM'].value_counts()

In [ ]:
df_rfm['Segmento_RFM'].nunique()

In [ ]:
df_rfm['Segmento_RFM'] = pd.to_numeric(df_rfm['Segmento_RFM'])

In [ ]:
df_rfm.info()

In [ ]:
df_rfm.describe().T

# Segmentação dos Clientes

Apesar de ter criado 32 segmentos, não vou segmentar os clientes nesses 32 grupos.

In [ ]:
df_final = df_rfm.copy()

## Função para criar os níveis (segmentação)

In [ ]:
def level(df):
    if ( (df['Segmento_RFM'] >= 424) | (df['Score_RFM'] >= 9) ) :
        return 'Clientes VIP'
    
    elif ((df['Score_RFM'] >= 8) & (df['M'] == 4)):
        return 'Clientes com Ticket Alto'
    
    elif ((df['Score_RFM'] >= 6) & (df['F'] >= 2)):
        return 'Clientes Leais'
    
    elif ((df['Score_RFM'] <= 4) & (df['R'] == 1)):
        return 'Clientes Quase Perdidos'
    
    elif ((df['Segmento_RFM'] >= 221) | (df['Score_RFM'] >= 6)):
        return 'Potenciais Clientes Leais'
    
    elif ((df['Segmento_RFM'] >= 121) & (df['R'] == 1) | (df['Score_RFM'] == 5)):
        return 'Clientes Que Precisam de Atenção'
    
    else:
        return 'Clientes Perdidos'

In [ ]:
df_final['Segmento_de_Clientes'] = df_final.apply(level, axis=1)

In [ ]:
df_final['Segmento_de_Clientes'].value_counts()

## Função para criar as ações de marketing

In [ ]:
def action(df):
    if ((df['Segmento_RFM'] >= 424) | (df['Score_RFM'] >= 9)) :
        return 'Oferecer edição limitada e programas de fidelidade'
    
    elif ((df['Score_RFM'] >= 8) & (df['M'] == 4)):
        return 'Oferecer itens mais caros (Upsell)'
    
    elif ((df['Score_RFM'] >= 6) & (df['F'] >= 2)):
        return 'Oferecer programas de fidelidade'
    
    elif ((df['Score_RFM'] <= 4) & (df['R'] == 1)):
        return 'Oferecer Incentivos de preços agressivos'
    
    elif ((df['Segmento_RFM'] >= 221) | (df['Score_RFM'] >= 6)):
        return 'Oferecer cupons de desconto'
    
    elif (((df['Segmento_RFM'] >= 121) & (df['R'] == 1)) | (df['Score_RFM'] == 5)):
        return 'Incentivos de preço e oferta por tempo limitado'
    
    else:
        return 'Não gaste muito tentando readquirir esse cliente'

In [ ]:
df_final['Marketing_Action'] = df_final.apply(action, axis=1)

In [ ]:
df_final['Marketing_Action'].value_counts()

In [ ]:
df_final

In [ ]:
rfm_segmentacao = pd.DataFrame(df_final.groupby(by = df_final['Segmento_de_Clientes']).agg({'recency': 'mean',
                                                                                'frequency': 'mean',
                                                                                'monetary': ['mean', 'count'],
                                                                                'Marketing_Action': 'unique'}))

In [ ]:
rfm_segmentacao = rfm_segmentacao.reset_index()
rfm_segmentacao

**Recomendações Para a Área de Negócio:**

**Clientes Leais** - São os clientes mais leais. Eles são ativos com compras frequentes e alto valor monetário. Eles podem ser os evangelistas da marca e a empresa deve se concentrar em servi-los muito bem. Eles podem ser os melhores clientes para obter feedback sobre o lançamento de qualquer novo produto ou ser os primeiros a adotar ou promover novos produtos/serviços.

**Potenciais Clientes Leais** - Alto potencial para entrar em nossos segmentos de clientes fiéis, por que não oferecer alguns brindes em sua próxima compra para mostrar que você os valoriza?

**Clientes Que Precisam de Atenção** - Mostrando sinais promissores com a quantidade e valor de sua compra, mas já faz um tempo que não compram. Vamos direcioná-los para seus itens da lista de desejos e um desconto com oferta por tempo limitado.

**Clientes Quase Perdidos** - Fizeram algumas compras iniciais, mas não voltaram desde então. Foi uma experiência ruim para o cliente? Ou adequação ao mercado do produto? Vamos gastar alguns recursos para construir o conhecimento de nossa marca com eles?

**Clientes Leais Que Compram com Frequência** - É sempre uma boa ideia tratar cuidadosamente todos os novos clientes, mas como esses clientes gastaram muito em suas compras, são ainda mais valiosos. É importante fazer com que eles se sintam valorizados e apreciados - e dar-lhes incentivos incríveis para continuar a interagir com a marca.

**Clientes VIP** - Buscam e querem mais do que preço. Programa de fidelidade ou produto/serviço exclusivo ou limitado são opções para manter esses clientes.

# Dashboard Interativo

In [ ]:
rfm_segmentacao

In [ ]:
rfm_segmentacao.shape

In [ ]:
# Multi-index
rfm_segmentacao.columns

In [ ]:
rfm_segmentacao['monetary']

In [ ]:
# Os valores que aparecerão no dashboard são o count
# Count ficou no multi-index do monetary, apesar de ser 'independente', é a contagem de cada grupo.
value_count_idx = rfm_segmentacao.columns[4]
value_count_idx

In [ ]:
fig = go.Figure(go.Treemap(
    labels = rfm_segmentacao['Segmento_de_Clientes'],
    parents = ['Segmento_de_Clientes', 
               'Segmento_de_Clientes', 
               'Segmento_de_Clientes', 
               'Segmento_de_Clientes', 
               'Segmento_de_Clientes', 
               'Segmento_de_Clientes', 
               'Segmento_de_Clientes'],  
    values = rfm_segmentacao[value_count_idx]))
fig.show()

# FIM